# F1 validation (2/2) - GCM runs: are the signals preserved by the downscaling?   [PLAN B]

The same F1 is applied to EC-Earth3 historical and to the four stabilization runs (notebook step2). This notebook checks
that the downscaling **preserves the signals** of the GCM:
- **climate change (CO2)**: 2025 original - historical, 2050 original - historical, 2050 original - 2025 original;
- **hosing**: 2025 hosing - original, 2050 hosing - original;

i.e. that the delta between two downscaled runs is the same as the delta between the two raw runs: maps, pixel-by-pixel
scatter (downscaled delta brought back to the EC-Earth3 grid against the raw delta: slope 1 = preserved), domain means,
annual cycle, extremes, dry days, and quantile by quantile (an analogue method cannot create values outside its training
range: deltas in the tails are the critical test).

Figures and tables: `outputs/` next to this notebook (not in git).

In [ ]:
import os
os.environ.setdefault("HDF5_USE_FILE_LOCKING", "FALSE")   # BDAP/EOS refuses HDF5 file locks ("NetCDF: HDF error")
import sys, warnings
from pathlib import Path
for _p in reversed((Path.cwd(), *Path.cwd().parents)):   # ba_utils.py, validation_utils.py: this folder or any parent folder
    sys.path.insert(0, str(_p))
import numpy as np, pandas as pd, xarray as xr
import matplotlib.pyplot as plt
import ba_utils as bu
import validation_utils as vu
warnings.filterwarnings("ignore")

CONFIG_PATH = None   # None = config_<profile>.json found automatically (written by notebook 00)
USE_DASK_CLUSTER = False   # True = use several CPU cores (recommended on a JEO-lab / JEO-desk node)
cfg = bu.load_config(CONFIG_PATH)
CLIENT = bu.start_dask() if USE_DASK_CLUSTER else None
VARS_ALL = cfg["variables"] + cfg.get("extra_variables", [])     # pr, tasmax, tasmin (+ rsds where it exists)

MASK_KIND = "buffer"          # "buffer" = LISFLOOD domain + 0.25 deg buffer | "domain" = without buffer (notebook 00c)
F1_TRAIN_PERIOD = ("1981", "2014")   # training period of the F1 to validate (notebook step1); its outputs: era5like_025/F1_<period>/
OVERWRITE_CACHE = False
DOMAIN = vu.init(cfg, MASK_KIND, borders_res="01M")
DS_DIR = bu.out_dir(cfg, "plan_B", "era5like_025", f"F1_{F1_TRAIN_PERIOD[0]}-{F1_TRAIN_PERIOD[1]}")   # outputs of step2
CACHE = bu.out_dir(cfg, "plan_B", "validation_cache")                 # statistics cache (EOS)
EXPLO_CACHE = bu.out_dir(cfg, "exploration_cache")                    # statistics already computed by 01_exploration
OUT = Path.cwd() / "outputs"                                          # figures + tables, next to this notebook (not in git)
OUT.mkdir(exist_ok=True)
print("downscaled outputs:", DS_DIR, "| figures:", OUT)

RUNS = {"gcm_historical": "historical", "gcm_2025_nohosing": "2025 original", "gcm_2025_hosing": "2025 hosing",
        "gcm_2050_nohosing": "2050 original", "gcm_2050_hosing": "2050 hosing"}
HIST_PERIOD = ("1981", "2014")        # historical run as in step1/step2 and in 01_exploration
PAIRS = [("gcm_2025_nohosing", "gcm_historical", "2025 original - historical"),
         ("gcm_2050_nohosing", "gcm_historical", "2050 original - historical"),
         ("gcm_2050_nohosing", "gcm_2025_nohosing", "2050 original - 2025 original"),
         ("gcm_2025_hosing", "gcm_2025_nohosing", "2025 hosing - original"),
         ("gcm_2050_hosing", "gcm_2050_nohosing", "2050 hosing - original")]

## 1. Data and statistics (cached): raw (native grid) and downscaled (0.25 deg) runs

In [ ]:
def read_downscaled(name, period=None):
    """Downscaled 'ERA5-like' 0.25 deg run written by step2: dict var -> DataArray (standard coordinates)."""
    f = DS_DIR / f"{name}_era5like_025.nc"
    if not f.exists():
        return {}
    ds = xr.open_dataset(f, chunks={"time": 31}, **bu._TIME_KW)
    out = {}
    for v in VARS_ALL:
        if v in ds:
            da = bu.norm_coords(ds[v]).transpose("time", "lat", "lon")
            out[v] = bu.subset_time(da, *period) if period else da
    return out


def read_raw(name, period=None, vars_=None):
    out = {}
    for v in (vars_ or VARS_ALL):
        if bu.dataset_available(cfg, name, v):
            da = bu.read_raw(cfg, name, v)
            out[v] = bu.subset_time(da, *period) if period else da
    return out


tagd = f"F1_{F1_TRAIN_PERIOD[0]}-{F1_TRAIN_PERIOD[1]}_{MASK_KIND}"
RAW, DSD = {}, {}
for name, lab in RUNS.items():
    per = HIST_PERIOD if name == "gcm_historical" else None
    d_ds = read_downscaled(name, per)
    if not d_ds:
        print(f"[missing] {lab}: no downscaled output in {DS_DIR} (step2)"); continue
    vars_ = [v for v in d_ds if bu.dataset_available(cfg, name, v)]
    p_explo = EXPLO_CACHE / f"{name}_stats_{MASK_KIND}.nc"
    if p_explo.exists() and not OVERWRITE_CACHE:
        RAW[name] = xr.open_dataset(p_explo).load()          # statistics of 01_exploration (same period, same domain)
    else:
        RAW[name] = vu.compute_stats(read_raw(name, per, vars_), CACHE / f"{name}_raw_stats_{MASK_KIND}.nc", OVERWRITE_CACHE, label=f"{lab} raw")
    DSD[name] = vu.compute_stats({v: d_ds[v] for v in vars_}, CACHE / f"{name}_downscaled_stats_{tagd}.nc", OVERWRITE_CACHE, label=f"{lab} downscaled")
print("runs:", [RUNS[n] for n in DSD])
any_ds = next(iter(DSD.values()))
GRID = any_ds[[k for k in any_ds.data_vars if k.endswith("_year_mean")][0]].isel(year=0, drop=True)   # ERA5 0.25 deg grid
any_raw = next(iter(RAW.values()))
GCM_GRID = any_raw[[k for k in any_raw.data_vars if k.endswith("_year_mean")][0]].isel(year=0, drop=True)
VPLOT = [v for v in ("pr", "tasmax", "tasmin", "dtr", "rsds") if any(vu.has(DSD[n], v) and vu.has(RAW[n], v) for n in DSD)]
PAIRS = [p for p in PAIRS if p[0] in DSD and p[1] in DSD]
print("pairs:", [p[2] for p in PAIRS], "| variables:", VPLOT)

## 2. Integrity of the downscaled runs

In [ ]:
rows = []
for name, s in DSD.items():
    for v in [x for x in VARS_ALL if f"period_{x}" in s.attrs]:
        rows.append({"run": RUNS[name], "var": v, "period": s.attrs[f"period_{v}"], "n days": s.attrs[f"ndays_{v}"],
                     "NaN frac": s.attrs[f"nan_frac_{v}"], "min": round(s.attrs[f"min_{v}"], 3), "max": round(s.attrs[f"max_{v}"], 3)})
    rows[-1]["frac tasmin > tasmax"] = s.attrs.get("frac_tasmin_gt_tasmax")
display(pd.DataFrame(rows).set_index(["run", "var"]))

## 3. Climatology of the downscaled runs (annual)

In [ ]:
fig, axes = vu.map_axes(len(VPLOT), len(DSD))
for r, v in enumerate(VPLOT):
    ref = vu.seas(DSD.get("gcm_historical", any_ds), v, "ANN") if vu.has(DSD.get("gcm_historical", any_ds), v) else None
    lim = vu.clim_lim(ref, v) if ref is not None else None
    for c, (name, s) in enumerate(DSD.items()):
        if vu.has(s, v):
            vu.draw(axes[r, c], vu.seas(s, v, "ANN"), f"{RUNS[name]} downscaled: {v}", cmap=vu.CMAP[v], lim=lim, label=vu.UNITS[v])
        else:
            axes[r, c].set_visible(False)
plt.tight_layout(); plt.savefig(OUT / "climatology_downscaled_runs.png", dpi=110); plt.show()

## 4. Delta maps: raw vs downscaled

For each pair: raw delta (EC-Earth3 grid, regridded to 0.25 deg for display), downscaled delta, and their difference.
pr in %. Dots: |Welch t| > 2 on the yearly values.

In [ ]:
def delta(a, b, v, season="ANN", kind="mean"):
    return vu.diff(vu.seas(a, v, season, kind), vu.seas(b, v, season, kind), v)

def to25(x):
    return bu.regrid_conservative(x, GRID.lat.values, GRID.lon.values).compute()

def to07(x):
    return bu.regrid_conservative(x, GCM_GRID.lat.values, GCM_GRID.lon.values).compute()

for a, b, lab in PAIRS:
    vs = [v for v in VPLOT if vu.has(RAW[a], v) and vu.has(RAW[b], v) and vu.has(DSD[a], v) and vu.has(DSD[b], v)]
    fig, axes = vu.map_axes(len(vs), 3)
    for r, v in enumerate(vs):
        d_raw, d_ds = delta(RAW[a], RAW[b], v), delta(DSD[a], DSD[b], v)
        lim = float(np.nanpercentile(np.abs(d_raw.where(vu.mask_for(d_raw))), 98)) or 1.0
        vu.draw(axes[r, 0], to25(d_raw), f"raw: {v}", cmap=vu.DCMAP[v], sym=True, lim=lim, label=vu.dlabel(v),
                sig=to25(vu.welch_sig(vu.yearly(RAW[a], v), vu.yearly(RAW[b], v)).astype(float)) > 0.5)
        vu.draw(axes[r, 1], d_ds, f"downscaled: {v}", cmap=vu.DCMAP[v], sym=True, lim=lim, label=vu.dlabel(v),
                sig=vu.welch_sig(vu.yearly(DSD[a], v), vu.yearly(DSD[b], v)))
        vu.draw(axes[r, 2], d_ds - to25(d_raw), f"downscaled - raw delta: {v}", cmap="PiYG", sym=True, lim=lim, label=vu.dlabel(v))
    fig.suptitle(lab, y=1.0)
    plt.tight_layout(); plt.savefig(OUT / f"delta_{a}_minus_{b}.png", dpi=110, bbox_inches="tight"); plt.show()

## 5. Is the delta preserved? Downscaled delta (back on the EC-Earth3 grid) against raw delta, pixel by pixel

Slope (area-weighted regression downscaled = a + slope x raw), correlation and ratio of the domain means.
**Preserved: slope ~ 1, corr ~ 1, mean ratio ~ 1.** Scatter plots: annual deltas.

In [ ]:
rows = []
for a, b, lab in PAIRS:
    vs = [v for v in VPLOT if vu.has(RAW[a], v) and vu.has(RAW[b], v) and vu.has(DSD[a], v) and vu.has(DSD[b], v)]
    fig, axes = plt.subplots(1, len(vs), figsize=(4.2 * len(vs), 4), squeeze=False)
    for c, v in enumerate(vs):
        for s in ("ANN", "DJF", "JJA"):
            d_raw = delta(RAW[a], RAW[b], v, s)
            d_ds7 = to07(delta(DSD[a], DSD[b], v, s)) if v != "pr" else \
                    vu.diff(to07(vu.seas(DSD[a], v, s)), to07(vu.seas(DSD[b], v, s)), v)
            rows.append({"pair": lab, "var": v, "season": s, **{k: round(x, 3) for k, x in vu.regression(d_raw, d_ds7).items()}})
            if s == "ANN":
                m = vu.mask_for(d_raw) & d_raw.notnull() & d_ds7.notnull()
                x, y = d_raw.where(m).values.ravel(), d_ds7.where(m).values.ravel()
                ok = np.isfinite(x) & np.isfinite(y)
                ax = axes[0, c]
                ax.scatter(x[ok], y[ok], s=2, alpha=0.3, color="tab:blue")
                lo, hi = np.nanpercentile(np.r_[x[ok], y[ok]], [0.5, 99.5])
                ax.plot([lo, hi], [lo, hi], color="k", lw=0.8, ls="--")
                ax.set_xlim(lo, hi); ax.set_ylim(lo, hi)
                ax.set_xlabel(f"raw delta ({vu.dlabel(v)})"); ax.set_ylabel("downscaled delta (on the EC-Earth3 grid)")
                ax.set_title(f"{lab}: {v} (slope {rows[-1]['slope']}, r {rows[-1]['corr']})", fontsize=8)
    plt.tight_layout(); plt.savefig(OUT / f"delta_scatter_{a}_minus_{b}.png", dpi=110); plt.show()
preserve = pd.DataFrame(rows).set_index(["pair", "var", "season"])
preserve.to_csv(OUT / "delta_preservation.csv")
display(preserve)

## 6. Deltas over the domain: annual cycle, raw vs downscaled

In [ ]:
fig, axes = plt.subplots(len(PAIRS), len(VPLOT), figsize=(4.5 * len(VPLOT), 3.2 * len(PAIRS)), squeeze=False)
for r, (a, b, lab) in enumerate(PAIRS):
    for c, v in enumerate(VPLOT):
        ax = axes[r, c]
        if not (vu.has(RAW[a], v, "clim_month") and vu.has(DSD[a], v, "clim_month") and vu.has(RAW[b], v, "clim_month")):
            ax.set_visible(False); continue
        for src, col, nm in ((RAW, "k", "raw"), (DSD, "tab:red", "downscaled")):
            ma, mb = vu.dmean(src[a][f"{v}_clim_month"]), vu.dmean(src[b][f"{v}_clim_month"])
            d = 100 * (ma - mb) / mb if v == "pr" else ma - mb
            ax.plot(d.month, d, "o-", ms=3, color=col, label=nm)
        ax.axhline(0, color="grey", lw=0.5); ax.set_title(f"{lab}: {v} ({vu.dlabel(v)})", fontsize=8)
axes[0, 0].legend(fontsize=7)
plt.tight_layout(); plt.savefig(OUT / "delta_annual_cycle.png", dpi=110); plt.show()

## 7. Deltas of the extremes (Rx1day, TXx, TNn), dry days and wet-day intensity: preserved?

Same regression as section 5, on the yearly extremes and on the dry-day / intensity indices.

In [ ]:
rows = []
INDICES = [(v, "ext", vu.EXT[v][1]) for v in vu.EXT] + [("pr", "dry", "dry-day fraction"), ("pr", "sdii", "wet-day intensity")]
for a, b, lab in PAIRS:
    for v, kind, name in INDICES:
        key = {"ext": f"{v}_year_ext", "dry": "pr_year_dry", "sdii": "pr_year_sdii"}[kind]
        if not all(key in s for s in (RAW[a], RAW[b], DSD[a], DSD[b])):
            continue
        ra, rb = vu.seas(RAW[a], v, "ANN", kind), vu.seas(RAW[b], v, "ANN", kind)
        da7, db7 = to07(vu.seas(DSD[a], v, "ANN", kind)), to07(vu.seas(DSD[b], v, "ANN", kind))
        if kind == "dry":
            d_raw, d_ds = 100 * (ra - rb), 100 * (da7 - db7)               # percentage points
        else:
            d_raw, d_ds = vu.diff(ra, rb, v), vu.diff(da7, db7, v)
        rows.append({"pair": lab, "index": name, **{k: round(x, 3) for k, x in vu.regression(d_raw, d_ds).items()}})
if rows:
    ext_tab = pd.DataFrame(rows).set_index(["pair", "index"])
    ext_tab.to_csv(OUT / "delta_preservation_extremes_drydays.csv")
    display(ext_tab)

# maps for the hosing pairs: delta of the extremes, raw vs downscaled
for a, b, lab in [p for p in PAIRS if p[0].endswith("_hosing") and p[1].endswith("_nohosing") and p[0][:8] == p[1][:8]]:
    evs = [v for v in vu.EXT if all(f"{v}_year_ext" in s for s in (RAW[a], RAW[b], DSD[a], DSD[b]))]
    if not evs:
        continue
    fig, axes = vu.map_axes(2, len(evs))
    for c, v in enumerate(evs):
        d_raw, d_ds = delta(RAW[a], RAW[b], v, kind="ext"), delta(DSD[a], DSD[b], v, kind="ext")
        lim = float(np.nanpercentile(np.abs(d_raw.where(vu.mask_for(d_raw))), 98)) or 1.0
        vu.draw(axes[0, c], to25(d_raw), f"{lab}, raw: {vu.EXT[v][1]}", cmap=vu.DCMAP[v], sym=True, lim=lim, label=vu.dlabel(v))
        vu.draw(axes[1, c], d_ds, f"{lab}, downscaled: {vu.EXT[v][1]}", cmap=vu.DCMAP[v], sym=True, lim=lim, label=vu.dlabel(v))
    plt.tight_layout(); plt.savefig(OUT / f"delta_extremes_{a}_minus_{b}.png", dpi=110); plt.show()

## 8. Deltas quantile by quantile (pooled over random pixels of the domain)

Delta of each quantile between the two runs, raw vs downscaled. If the downscaled delta shrinks in the upper tail
(e.g. q99 of tasmax), the analogue method is limited by its training range (it cannot produce values never seen in
ERA5 1981-2014): a known limitation of analogues in a warmer climate.

In [ ]:
keys = [k for k in ("pr_q_wet", "tasmax_q", "tasmin_q", "dtr_q", "rsds_q")]
fig, axes = plt.subplots(len(PAIRS), len(keys), figsize=(4.2 * len(keys), 3.2 * len(PAIRS)), squeeze=False)
rows = []
for r, (a, b, lab) in enumerate(PAIRS):
    for c, k in enumerate(keys):
        ax = axes[r, c]
        if not all(k in s for s in (RAW[a], RAW[b], DSD[a], DSD[b])):
            ax.set_visible(False); continue
        rel = k.startswith("pr")
        dq = {nm: (100 * (src[a][k] - src[b][k]) / src[b][k] if rel else src[a][k] - src[b][k]).values
              for nm, src in (("raw", RAW), ("downscaled", DSD))}
        ax.plot(vu.QUANTILES, dq["raw"], "o-", ms=3, color="k", label="raw")
        ax.plot(vu.QUANTILES, dq["downscaled"], "s--", ms=3, color="tab:red", label="downscaled")
        ax.set_xscale("logit"); ax.axhline(0, color="grey", lw=0.5)
        ax.set_title(f"{lab}: {k.replace('_q_wet', ' wet').replace('_q', '')} ({'%' if rel else 'abs.'})", fontsize=8)
        for qq in (0.5, 0.99):
            i = vu.QUANTILES.index(qq)
            rows.append({"pair": lab, "variable": k.replace("_q_wet", " wet").replace("_q", ""), "quantile": qq,
                         "delta raw": round(float(dq["raw"][i]), 3), "delta downscaled": round(float(dq["downscaled"][i]), 3)})
axes[0, 0].legend(fontsize=7)
plt.tight_layout(); plt.savefig(OUT / "delta_quantiles.png", dpi=110); plt.show()
qd = pd.DataFrame(rows).set_index(["pair", "variable", "quantile"]); qd["ratio"] = (qd["delta downscaled"] / qd["delta raw"]).round(3)
qd.to_csv(OUT / "delta_quantiles.csv"); display(qd)

## Conclusions (to fill in)

- CO2 delta (2025 / 2050 original - historical) preserved? (slope, corr, mean ratio): ...
- Hosing delta preserved? ...
- Extremes / dry days / tails (q99): ...
- Limits of the analogue method in the warmer / hosing climates: ...